In [ ]:
import pandas as pd
import requests
import time

# CSVを読み込む
df = pd.read_csv("../data/災害時における医療/08medcareindisasters260325.csv")

GOOGLE_API_KEY = "***********"

def geocode(hospital_name, area):
    """病院名＋地域名で住所と緯度経度を取得"""
    query = f"{hospital_name} 静岡県 {area}"
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {
        "address": query,
        "key": GOOGLE_API_KEY,
        "language": "ja",
        "region": "jp"
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
        if data["status"] == "OK":
            result = data["results"][0]
            address = result["formatted_address"]
            lat = result["geometry"]["location"]["lat"]
            lng = result["geometry"]["location"]["lng"]
            return address, lat, lng
        else:
            return None, None, None
    except Exception:
        return None, None, None

addresses, lats, lngs = [], [], []

for _, row in df.iterrows():
    addr, lat, lng = geocode(row["hospital_name"], row["area"])
    addresses.append(addr)
    lats.append(lat)
    lngs.append(lng)
    time.sleep(0.1)  # API rate limit対策

df["address"] = addresses
df["latitude"] = lats
df["longitude"] = lngs

df.to_csv("../data/災害時における医療/08medcareindisasters_geocoded.csv", index=False, encoding="utf-8-sig")
print("完了:", len(df), "件")
print(df[["hospital_name", "address", "latitude", "longitude"]].head())

完了: 178 件
          hospital_name                        address   latitude   longitude
0                     ―          日本、〒411-0023 静岡県三島市加茂  35.140834  138.939177
1                伊東市民病院      日本、〒414-0055 静岡県伊東市岡１９６−１  34.960398  139.087837
2          国際医療福祉大学熱海病院    日本、〒413-0012 静岡県熱海市東海岸町１３−１  35.103678  139.083559
3  独立行政法人国立病院機構静岡医療センター  日本、〒411-8611 静岡県駿東郡清水町長沢７６２−１  35.100095  138.890144
4                沼津市立病院   日本、〒410-0302 静岡県沼津市東椎路春ノ木５５０  35.119769  138.835208


In [4]:
import pandas as pd
import json

df = df[df["latitude"].notna() & df["longitude"].notna()].copy()

color_map = {
    "災害拠点病院": "#e53e3e",
    "災害拠点精神科病院": "#9f7aea",
    "救護病院": "#38a169",
    "静岡DMAT指定病院": "#dd6b20",
    "応援班設置病院（普通班）": "#3182ce",
    "応援班設置病院（精神科班）": "#b83280",
}

records = []
for _, r in df.iterrows():
    records.append({
        "name": r["hospital_name"],
        "area": r["area"],
        "section": r["section"],
        "subtype": str(r["subtype"]),
        "lat": r["latitude"],
        "lng": r["longitude"],
        "color": color_map.get(str(r["subtype"]), "#718096"),
    })

geojson_str = json.dumps(records, ensure_ascii=False)

subtypes_def = [
    {"key": "災害拠点病院", "color": "#e53e3e"},
    {"key": "災害拠点精神科病院", "color": "#9f7aea"},
    {"key": "救護病院", "color": "#38a169"},
    {"key": "静岡DMAT指定病院", "color": "#dd6b20"},
    {"key": "応援班設置病院（普通班）", "color": "#3182ce"},
    {"key": "応援班設置病院（精神科班）", "color": "#b83280"},
]
subtypes_str = json.dumps(subtypes_def, ensure_ascii=False)

html = f"""<!DOCTYPE html>
<html lang="ja">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>静岡県 災害時医療機関マップ</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
* {{ box-sizing: border-box; margin: 0; padding: 0; }}
body {{ font-family: 'Hiragino Sans', 'Meiryo', sans-serif; background: #f0f4f8; }}
#header {{ background: #1a365d; color: white; padding: 14px 20px; position: relative; z-index: 1000; box-shadow: 0 2px 8px rgba(0,0,0,.2); }}
#header h1 {{ font-size: 1.1rem; font-weight: 700; }}
#header span {{ font-size: .8rem; opacity: .75; }}
#controls {{ position: absolute; top: 70px; right: 12px; z-index: 1000; background: white; border-radius: 10px; box-shadow: 0 4px 16px rgba(0,0,0,.12); padding: 14px 16px; min-width: 230px; }}
#controls h3 {{ font-size: .78rem; font-weight: 700; color: #4a5568; text-transform: uppercase; letter-spacing: .08em; margin-bottom: 10px; }}
.legend-item {{ display: flex; align-items: center; gap: 8px; margin-bottom: 7px; cursor: pointer; padding: 4px 6px; border-radius: 6px; transition: background .15s; user-select: none; }}
.legend-item:hover {{ background: #edf2f7; }}
.legend-item.disabled {{ opacity: .35; }}
.dot {{ width: 14px; height: 14px; border-radius: 50%; flex-shrink: 0; border: 2px solid rgba(255,255,255,.8); box-shadow: 0 1px 3px rgba(0,0,0,.2); }}
.legend-label {{ font-size: .82rem; line-height: 1.3; }}
.count {{ margin-left: auto; font-size: .75rem; color: #a0aec0; font-weight: 600; }}
#stats {{ margin-top: 10px; padding-top: 10px; border-top: 1px solid #e2e8f0; font-size: .78rem; color: #718096; }}
#map {{ height: calc(100vh - 56px); width: 100%; }}
.popup-title {{ font-weight: 700; font-size: .92rem; margin-bottom: 4px; color: #1a365d; }}
.popup-sub {{ font-size: .78rem; color: #4a5568; }}
.popup-badge {{ display: inline-block; padding: 2px 8px; border-radius: 99px; font-size: .72rem; font-weight: 600; color: white; margin-top: 5px; }}
</style>
</head>
<body>
<div id="header">
  <h1>🏥 静岡県 災害時医療機関マップ</h1>
  <span>静岡県保健医療計画【R8.3月末時点】</span>
</div>
<div id="controls">
  <h3>凡例（クリックで表示切替）</h3>
  <div id="legend"></div>
  <div id="stats"></div>
</div>
<div id="map"></div>
<script>
const data = {geojson_str};
const subtypes = {subtypes_str};

const map = L.map('map', {{center: [34.97, 138.38], zoom: 9}});
L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{
  attribution: '© OpenStreetMap contributors',
  maxZoom: 18
}}).addTo(map);

const layers = {{}};
const vis = {{}};
subtypes.forEach(s => {{
  layers[s.key] = [];
  vis[s.key] = true;
}});

data.forEach(d => {{
  const icon = L.divIcon({{
    className: '',
    html: `<div style="width:12px;height:12px;border-radius:50%;background:${{d.color}};border:2.5px solid white;box-shadow:0 2px 5px rgba(0,0,0,.3)"></div>`,
    iconSize: [12, 12],
    iconAnchor: [6, 6]
  }});

  const m = L.marker([d.lat, d.lng], {{icon}})
    .bindPopup(`
      <div class="popup-title">${{d.name}}</div>
      <div class="popup-sub">２次医療圏：${{d.area}}</div>
      <div class="popup-sub">区分：${{d.section}}</div>
      <span class="popup-badge" style="background:${{d.color}}">${{d.subtype}}</span>
    `);

  m.addTo(map);
  if (layers[d.subtype]) layers[d.subtype].push(m);
}});

const legend = document.getElementById('legend');
subtypes.forEach(s => {{
  const cnt = (layers[s.key] || []).length;
  if (!cnt) return;

  const el = document.createElement('div');
  el.className = 'legend-item';
  el.innerHTML = `<div class="dot" style="background:${{s.color}}"></div><span class="legend-label">${{s.key}}</span><span class="count">${{cnt}}</span>`;

  el.addEventListener('click', () => {{
    vis[s.key] = !vis[s.key];
    el.classList.toggle('disabled', !vis[s.key]);
    (layers[s.key] || []).forEach(m => vis[s.key] ? map.addLayer(m) : map.removeLayer(m));
  }});

  legend.appendChild(el);
}});

document.getElementById('stats').textContent = `合計 ${{data.length}} 機関`;
</script>
</body>
</html>"""

with open("../data/災害時における医療/python_hospital_map.html", "w", encoding="utf-8") as f:
    f.write(html)

print("../data/災害時における医療/python_hospital_map.html を出力しました")

../data/災害時における医療/python_hospital_map.html を出力しました
